ネコがベッドにのった。ネコは<そこで>眠った

In [ ]:
import torch
from torch import nn

In [ ]:
torch.manual_seed(0)   # 乱数の出発点を固定

words = ["ベッド", "ネコ", "のった", "そこ"]
x = torch.eye(4)  # # 各単語を区別する4次元ベクトル
print(x)

tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])


In [ ]:
# "ベッド", "ネコ", "乗った", "そこ"
# 入力ノード：4個 — x1、x2、x3、x4
# 出力ノード：2個 — 出力値が q1、q2 仮定
# 重み W1, w2, w3, w4, w5, w6, w7, w8

W_Q = nn.Linear(4, 2, bias=False)
W_K = nn.Linear(4, 2, bias=False)

「そこ」と各単語の関連スコアを計算する

In [ ]:
def calculate_scores():
    q = W_Q(x[3])
    k = W_K(x)
    return (k @ q) / (2 ** 0.5)

In [ ]:
target = torch.tensor([0])

# 重みの調整方法と、誤差の計算方法を用意する
optimizer = torch.optim.Adam(
    list(W_Q.parameters()) + list(W_K.parameters()),
    lr=0.05
)

# 誤差を計算する
loss_fn = nn.CrossEntropyLoss()

# ---- 教師無し ----
# 配分を表示する関数
def show_weights():
    with torch.no_grad():
        weights = torch.softmax(calculate_scores(), dim=0)
    for word, weight in zip(words, weights):
        print(f"{word}:{weight.item():.3f}")

print("【学習前】")
show_weights()

# -----  学習　-------
# 繰り返しで少しずつ誤差を小さくする
# ⑤ ベッドへの配分が高くなるように学習する
for step in range(100):
    scores = calculate_scores()
    loss = loss_fn(scores.unsqueeze(0), target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("\n【学習後】")
show_weights()

【学習前】
ベッド:0.234
ネコ:0.233
のった:0.260
そこ:0.273

【学習後】
ベッド:0.999
ネコ:0.000
のった:0.000
そこ:0.000
